# 국어나래 코치 모델 다시 내보내기 (Gemma 3 1B → 앱용 ONNX)

DPO로 학습해 병합해 둔 모델(`MERGED_REPO`)을 받아서, 앱이 읽는 **ONNX 파일 하나(약 951MB)** 로 다시 만들고 `REPO`에 올려요. 학습은 하지 않아요.

- 방식: 대부분의 층은 동적 int8, `down_proj`는 GPTQ 4비트(처음·마지막 3층은 8비트). 안전 동작이 무너지지 않는 조합으로 확인한 기본값이에요.
- 올리면 `REPO`의 옛 `onnx/model_int8.onnx`가 새 파일로 바뀌어요(`redteam/` 폴더는 그대로 둬요).

## Kaggle 설정
1. 이 노트북을 가져오기(`File → Import Notebook → GitHub` 또는 파일 올리기)
2. 오른쪽 `Settings`에서 `Internet`을 **켜기**(전화번호 인증 필요). Input 첨부는 필요 없어요.
3. `Accelerator`는 **None이어도, GPU여도 돼요.** 이 변환은 CPU로 계산해서 GPU를 쓰지 않아요(GPU 시간만 쓰므로 **None이 낫고 속도는 같아요**).
4. `Add-ons → Secrets`에 `HF_TOKEN`(Write 권한)을 추가하고 이 노트북에 연결
5. `Run All`로 위에서부터 실행(오래 걸리니 `Save & Run All`도 좋아요)

## 걸리는 시간(대략)
받기 몇 분 + 변환 환경 만들기 5~10분 + 변환·GPTQ 계산 30~60분(CPU 코어가 적을수록 오래 걸려요) + 올리기 5~10분. 모두 합쳐 **1시간 안팎**이에요.

## 잘 끝났는지 보는 법(로그 한 줄씩)
- `[확인] 가중치가 모두 맞게 읽혔어요.`
- `[확인] fp32 ONNX 다음 토큰=…` (실패하면 `[변환 중단]`이 떠요)
- `완료: export/narae-gemma-1b  (onnx/model_int8.onnx 951MB)` 비슷한 크기(약 950MB)
- 6칸의 `sha256`과 크기, 마지막 칸의 `완료: https://huggingface.co/…` 주소

**끝나면 Claude에게 알려 주세요**(배포 파일로 레드팀 950문항을 다시 확인한 뒤 앱 기본 모델로 바꿉니다).

## 설정

In [ ]:
# ── 설정 ──
REPO = 'LUDICERECIDUL/narae-coach-gemma3-1b'                 # 앱이 받는 모델 저장소(여기에 올려요)
MERGED_REPO = 'LUDICERECIDUL/narae-coach-gemma3-1b-merged'   # 병합한 DPO 가중치(여기서 받아요)
PRIVATE = False            # 이미 있는 저장소의 공개 여부는 바꾸지 않아요.

## 0. 준비: 코드 받기, 토큰 확인

In [ ]:
import os, sys, shutil, glob, json, subprocess

# 명령이 실패하면 '모두 실행'이 거기서 멈추도록(그냥 !python 은 실패해도 다음 칸으로 넘어가요)
def sh(cmd):
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(f'명령이 실패했어요(코드 {proc.returncode}): {cmd}\n위 오류 메시지 마지막 부분을 Claude에게 알려 주세요.')

# 1) 어디서 돌고 있는지 알아내고 HF 토큰을 읽습니다(Kaggle 보안 비밀 / Colab 보안 비밀 / 환경 변수).
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if os.path.exists('/kaggle'):
    PLATFORM = 'kaggle'
    CODE_ROOT = '/kaggle/working'
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = (UserSecretsClient().get_secret('HF_TOKEN') or HF_TOKEN).strip()
    except Exception as e:
        print('Kaggle 보안 비밀(HF_TOKEN)을 못 읽었어요:', type(e).__name__, e)
else:
    try:
        from google.colab import userdata
        PLATFORM = 'colab'
        CODE_ROOT = '/content'
        try:
            HF_TOKEN = (userdata.get('HF_TOKEN') or HF_TOKEN).strip()
        except Exception as e:
            print('Colab 보안 비밀(HF_TOKEN)을 못 읽었어요:', type(e).__name__, e)
    except Exception:
        PLATFORM = 'local'
        CODE_ROOT = os.getcwd()
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass('Hugging Face 토큰(Write 권한)을 붙여 넣으세요: ').strip()
assert HF_TOKEN, 'HF_TOKEN이 필요해요(올리기에 써요).'
os.environ['HF_TOKEN'] = HF_TOKEN
print('실행 환경:', PLATFORM)

# 2) 코드 받기(공개 저장소). 이미 받았으면 최신본만 받아요.
os.chdir(CODE_ROOT)
if os.path.isdir('narae-kor-aidt'):
    sh('cd narae-kor-aidt && rm -rf training/__pycache__ && git pull -q')
else:
    sh('git clone -q --depth 1 -b feat/module-core https://github.com/oswprogram10-tech/narae-kor-aidt.git')
TRAIN = os.path.join(CODE_ROOT, 'narae-kor-aidt', 'training')
os.chdir(TRAIN)
print('작업 폴더:', TRAIN)
print('CPU 코어 수:', os.cpu_count())

## 1. 병합 모델 받기
`MERGED_REPO`(공개)에서 가중치와 설정을 `merged/` 폴더로 받아요(약 2GB).

In [ ]:
import time
from huggingface_hub import snapshot_download
t0 = time.time()
MERGED_DIR = os.path.join(TRAIN, 'merged')
snapshot_download(repo_id=MERGED_REPO, local_dir=MERGED_DIR, token=HF_TOKEN or None)
assert glob.glob(os.path.join(MERGED_DIR, '*.safetensors')), f'{MERGED_REPO} 에서 *.safetensors 를 받지 못했어요'
print(f'받기 끝: {MERGED_DIR} ({time.time() - t0:.0f}초)')
sh(f'ls -la {MERGED_DIR}')

## 2. 변환 환경 만들기 + ONNX 변환
변환 도구는 학습 환경과 버전이 달라서 따로 가상환경(`/tmp/onnxenv`)을 만들어요. 기본 방식(동적 int8 + `down_proj` GPTQ 4비트, 처음·마지막 3층 8비트)으로 `export/narae-gemma-1b`에 만들어요.
GPTQ 보정은 CPU로 계산해요(GPU를 쓰지 않아요). 로그가 한참 천천히 올라와도 정상이에요.

In [ ]:
t0 = time.time()
sh('bash export_env.sh /tmp/onnxenv')
print(f'[시간] 변환 환경: {time.time() - t0:.0f}초')
t1 = time.time()
sh(f'rm -rf export/narae-gemma-1b && /tmp/onnxenv/bin/python -u export_onnx.py --model {MERGED_DIR} --out export/narae-gemma-1b')
print(f'[시간] 변환: {time.time() - t1:.0f}초')

## 3. 결과 확인(크기·sha256·파일 목록)

In [ ]:
import hashlib
OUT = 'export/narae-gemma-1b'
P = os.path.join(OUT, 'onnx', 'model_int8.onnx')
assert os.path.exists(P), '변환 결과(onnx/model_int8.onnx)가 없어요'
size = os.path.getsize(P)
h = hashlib.sha256()
with open(P, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
print(f'model_int8.onnx  크기: {size:,} 바이트 ({size / 1e6:.0f}MB)')
print('sha256:', h.hexdigest())
assert 900e6 < size < 1000e6, '크기가 예상(약 951MB)과 많이 달라요. 올리지 말고 Claude에게 알려 주세요.'
sh(f'ls -laR {OUT}')

## 4. Hugging Face에 올리기
`export/narae-gemma-1b` 폴더를 `REPO`에 올려요. 같은 이름의 파일은 새 것으로 바뀌고, `REPO`의 `redteam/` 폴더는 건드리지 않아요.

In [ ]:
msg = '국어나래 코치 모델: 동적 int8 + down_proj GPTQ (951MB)'
sh(f'python upload_hf.py --repo {REPO} --folder export/narae-gemma-1b --names "" --message "{msg}"' + (' --private' if PRIVATE else ''))